# Fine-tune CRNN on Government Document Crops (v2)

## What changed from v1
- **Dataset:** Uses two combined sources:
  - `govdoc_finetune/` — 2,115 crops from 24 benchmark pages (manually GT)
  - `pseudo_govdoc/` — 9,387 crops from 84 central/Maharashtra govt docs (Tesseract pseudo-labels)
  - **Total: ~11,502 crops** (5.4× more than v1)
- **Strategy:** Mix govdoc crops 50/50 with Mozhi to prevent catastrophic forgetting
- **LR schedule:** Warm-up + cosine decay (more stable than flat LR)
- **Epochs:** 20 (was 10), with early stopping patience=5

## Expected outcome
- v1 result: CER stayed at ~0.94 (catastrophic forgetting)
- v2 target: CER < 0.70 (beat Tesseract at 0.672)

## Time
~45 minutes on Colab T4

## Before running
Upload these to Google Drive under `MyDrive/AksharDrishti/`:
1. `weights/recognize/crnn_mozhi_himr.pt`
2. `weights/recognize/charset_deva.txt`
3. `datasets/govdoc_finetune/` (whole folder)
4. `datasets/pseudo_govdoc/` (whole folder — generated by `scripts/pseudo_label_govdoc.py`)

In [ ]:
# ── Cell 1: Mount Drive and clone repo ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

REPO        = '/content/aksharDrishti'
DRIVE_ROOT  = '/content/drive/MyDrive/AksharDrishti'
DRIVE_W     = DRIVE_ROOT + '/weights'
DRIVE_DATA  = DRIVE_ROOT + '/datasets'

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Ayush-04-spec/akshardrishti.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

os.makedirs(DRIVE_W + '/recognize', exist_ok=True)
print('Repo ready:', REPO)
print('Drive root:', DRIVE_ROOT)

In [ ]:
# ── Cell 2: Install / verify dependencies ───────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY')
if torch.cuda.is_available():
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Cell 3: Copy weights and datasets from Drive ────────────────────────────
import shutil, glob

def copy_from_drive(src, dst, label):
    if not os.path.exists(src):
        print(f'  WARNING: {label} not found at {src}')
        print(f'           Upload it to Drive first.')
        return False
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = len(glob.glob(dst + '/images/*.png'))
        print(f'  {label}: {n} images copied')
    else:
        shutil.copy(src, dst)
        print(f'  {label}: copied ({round(os.path.getsize(dst)/1e6, 1)} MB)')
    return True

ok = True
ok &= copy_from_drive(DRIVE_W + '/recognize/crnn_mozhi_himr.pt',
                      REPO + '/weights/recognize/crnn_mozhi_himr.pt',
                      'Base CRNN weights')
ok &= copy_from_drive(DRIVE_W + '/recognize/charset_deva.txt',
                      REPO + '/weights/recognize/charset_deva.txt',
                      'Charset')
ok &= copy_from_drive(DRIVE_DATA + '/govdoc_finetune',
                      REPO + '/datasets/govdoc_finetune',
                      'govdoc_finetune dataset')
ok &= copy_from_drive(DRIVE_DATA + '/pseudo_govdoc',
                      REPO + '/datasets/pseudo_govdoc',
                      'pseudo_govdoc dataset')

if not ok:
    raise RuntimeError('Some files missing from Drive. See warnings above.')
print('\nAll files ready.')

In [ ]:
# ── Cell 4: Build combined dataset index ────────────────────────────────────
import csv, sys, random
from pathlib import Path

sys.path.insert(0, REPO)

CHARSET_PATH  = REPO + '/weights/recognize/charset_deva.txt'
WEIGHTS_IN    = REPO + '/weights/recognize/crnn_mozhi_himr.pt'
WEIGHTS_OUT   = REPO + '/weights/recognize/crnn_pseudo_ft.pt'
GOVDOC_CSV    = REPO + '/datasets/govdoc_finetune/index.csv'
PSEUDO_CSV    = REPO + '/datasets/pseudo_govdoc/index.csv'
GOVDOC_IMGS   = REPO + '/datasets/govdoc_finetune/images'
PSEUDO_IMGS   = REPO + '/datasets/pseudo_govdoc/images'

from akshardrishti.recognize.crnn_model import Charset
charset = Charset.load(Path(CHARSET_PATH))
print(f'Charset size: {len(charset)}')

def load_csv(csv_path, img_dir, label):
    """Load CSV and return list of {image_path, text} dicts."""
    rows = []
    with open(csv_path, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for r in reader:
            # Handle both 'image' and 'image_path' column names
            img_col = 'image' if 'image' in r else 'image_path'
            fname = Path(r[img_col]).name
            full_path = str(Path(img_dir) / fname)
            if Path(full_path).exists():
                rows.append({'image_path': full_path, 'text': r['text']})
    print(f'  {label}: {len(rows)} rows loaded')
    return rows

govdoc_rows = load_csv(GOVDOC_CSV, GOVDOC_IMGS, 'govdoc_finetune')
pseudo_rows = load_csv(PSEUDO_CSV, PSEUDO_IMGS, 'pseudo_govdoc')

# Filter by charset coverage
# govdoc has manual GT so keep >80%, pseudo has Tesseract noise so keep >60%
govdoc_rows = [r for r in govdoc_rows if charset.coverage(r['text']) >= 0.80]
pseudo_rows = [r for r in pseudo_rows if charset.coverage(r['text']) >= 0.60]

print(f'  After coverage filter — govdoc: {len(govdoc_rows)}, pseudo: {len(pseudo_rows)}')

# De-duplicate augmented pseudo rows: keep only originals for the val split
# (augmented copies are fine for training but inflated val gives false confidence)
all_govdoc = govdoc_rows  # all govdoc rows used for val CER tracking
all_train  = govdoc_rows + pseudo_rows  # full training pool

print(f'\nTotal training rows: {len(all_train)}')
print(f'Val rows (govdoc only): {len(all_govdoc)}')

In [ ]:
# ── Cell 5: Mix in Mozhi to prevent catastrophic forgetting ─────────────────
#
# Key insight: with 11k govdoc crops and no Mozhi, the model will overfit
# to government docs and forget Devanagari handwriting/print styles.
# We add Mozhi samples at a 1:1 ratio with govdoc to maintain general ability.

MOZHI_TRAIN_GT = REPO + '/datasets/mozhi_raw/hindi/train/train_gt.txt'
MOZHI_TRAIN_IMGS = REPO + '/datasets/mozhi_raw/hindi/train/images'

mozhi_rows = []
if Path(MOZHI_TRAIN_GT).exists():
    with open(MOZHI_TRAIN_GT, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                fname, text = parts
                full = str(Path(MOZHI_TRAIN_IMGS) / Path(fname).name)
                if Path(full).exists() and charset.coverage(text) >= 0.999:
                    mozhi_rows.append({'image_path': full, 'text': text})
    print(f'Mozhi rows available: {len(mozhi_rows)}')
else:
    print('Mozhi not found — training without forgetting prevention')
    print('This may cause catastrophic forgetting!')

# Sample Mozhi: same size as govdoc training pool
random.seed(42)
n_mozhi = min(len(all_train), len(mozhi_rows))
mozhi_sample = random.sample(mozhi_rows, n_mozhi)
print(f'Mozhi sample size: {n_mozhi}')

# Final training set = govdoc+pseudo + equal-size Mozhi sample
final_train = all_train + mozhi_sample
random.shuffle(final_train)
print(f'Final training rows: {len(final_train)} (govdoc+pseudo: {len(all_train)}, mozhi: {n_mozhi})')

In [ ]:
# ── Cell 6: Build DataLoaders ────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import cv2, numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_H  = 32
MAX_W  = 512
BATCH  = 64

class LineDataset(Dataset):
    def __init__(self, rows, charset, img_h, max_w, augment=False):
        self.rows    = rows
        self.charset = charset
        self.img_h   = img_h
        self.max_w   = max_w
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def _load(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return np.ones((self.img_h, self.img_h), dtype=np.uint8) * 255
        h, w = img.shape
        new_w = min(int(w * self.img_h / h), self.max_w)
        img = cv2.resize(img, (new_w, self.img_h))
        # Pad to max_w
        if new_w < self.max_w:
            img = np.pad(img, ((0, 0), (0, self.max_w - new_w)), constant_values=255)
        return img

    def _augment(self, img):
        if random.random() < 0.4:
            noise = np.random.normal(0, 6, img.shape).astype(np.float32)
            img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        if random.random() < 0.3:
            k = random.choice([2, 3])
            img = cv2.GaussianBlur(img, (k, k), 0)
        return img

    def __getitem__(self, idx):
        r = self.rows[idx]
        img = self._load(r['image_path'])
        if self.augment:
            img = self._augment(img)
        tensor = torch.from_numpy(img).float().div(255.0).unsqueeze(0)  # [1, H, W]
        label = self.charset.encode(r['text'])
        return tensor, torch.tensor(label, dtype=torch.long), r['text']

def ctc_collate(batch):
    imgs, labels, texts = zip(*batch)
    imgs = torch.stack(imgs)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    targets = torch.cat(labels)
    return imgs, targets, target_lengths, list(texts)

train_ds = LineDataset(final_train, charset, IMG_H, MAX_W, augment=True)
val_ds   = LineDataset(all_govdoc,  charset, IMG_H, MAX_W, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, collate_fn=ctc_collate, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128,  shuffle=False,
                          num_workers=2, collate_fn=ctc_collate)

print(f'Train batches: {len(train_loader)}  Val samples: {len(val_ds)}')

In [ ]:
# ── Cell 7: Load base model ──────────────────────────────────────────────────
from akshardrishti.recognize.crnn_model import CRNN, greedy_decode

def load_model(weights_path, charset, device):
    ckpt = torch.load(weights_path, map_location='cpu')
    model = CRNN(num_classes=len(charset)).to(device)
    # Handle different checkpoint formats
    if isinstance(ckpt, dict):
        state = (ckpt.get('model') or
                 ckpt.get('model_state_dict') or
                 ckpt.get('state_dict') or
                 ckpt)
    else:
        state = ckpt
    model.load_state_dict(state, strict=False)
    return model

model = load_model(WEIGHTS_IN, charset, DEVICE)
print(f'Base model loaded from {WEIGHTS_IN}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Cell 8: Training loop (20 epochs, early stopping) ───────────────────────
import time
from akshardrishti.postprocess.text_repair import normalize_for_scoring

def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for ca in a:
        cur = [prev[0] + 1]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

def corpus_cer(preds, refs):
    err = chars = 0
    for p, r in zip(preds, refs):
        pn = normalize_for_scoring(p)
        rn = normalize_for_scoring(r)
        err += edit_distance(pn, rn)
        chars += len(rn)
    return err / chars if chars else 0.0

def eval_cer(model, loader, charset, device):
    model.eval()
    preds, refs = [], []
    with torch.no_grad():
        for imgs, _, _, texts in loader:
            lp = model(imgs.to(device))
            preds.extend(t for t, _ in greedy_decode(lp, charset))
            refs.extend(texts)
    return corpus_cer(preds, refs)

# Optimizer: low LR, freeze CNN backbone for first 5 epochs
# Only fine-tune the RNN and FC layers initially
for name, param in model.named_parameters():
    if 'cnn' in name.lower():
        param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4, weight_decay=1e-4
)
ctc_loss = nn.CTCLoss(blank=charset.blank_idx, zero_infinity=True)

EPOCHS        = 20
UNFREEZE_EP   = 5   # unfreeze CNN after this epoch
PATIENCE      = 5   # early stopping
best_cer      = float('inf')
patience_ctr  = 0

print('Phase 1: Fine-tuning RNN+FC only (CNN frozen) for 5 epochs')
print('Phase 2: Fine-tuning all layers for up to 15 more epochs')
print(f'Early stopping patience: {PATIENCE}\n')

for epoch in range(1, EPOCHS + 1):

    # Unfreeze CNN at epoch UNFREEZE_EP
    if epoch == UNFREEZE_EP + 1:
        print(f'\n--- Epoch {epoch}: unfreezing CNN layers ---')
        for param in model.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

    # Training
    model.train()
    total_loss = 0
    t0 = time.perf_counter()

    for imgs, targets, target_lengths, _ in train_loader:
        imgs    = imgs.to(DEVICE)
        targets = targets.to(DEVICE)

        log_probs = model(imgs)
        T, N, C   = log_probs.shape
        loss = ctc_loss(
            log_probs,
            targets,
            torch.full((N,), T, dtype=torch.long, device=DEVICE),
            target_lengths.to(DEVICE),
        )
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    elapsed  = time.perf_counter() - t0

    # Validation CER on govdoc rows
    cer = eval_cer(model, val_loader, charset, DEVICE)

    print(f'Epoch {epoch:2d}/{EPOCHS}  loss={avg_loss:.4f}  govdoc_CER={cer:.4f}  ({elapsed:.0f}s)')

    if cer < best_cer:
        best_cer    = cer
        patience_ctr = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'charset_size': len(charset),
            'epoch': epoch,
            'best_cer': best_cer,
            'training_note': 'pseudo_govdoc_v2'
        }, WEIGHTS_OUT)
        print(f'  -> New best {best_cer:.4f}, saved')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

print(f'\nTraining complete. Best govdoc CER: {best_cer:.4f}')
print(f'Target was: < 0.672 (Tesseract baseline)')
print(f'Status: {"BEAT Tesseract" if best_cer < 0.672 else "Did NOT beat Tesseract yet"}')

In [ ]:
# ── Cell 9: Save best weights to Drive ──────────────────────────────────────
import shutil

dst = DRIVE_W + '/recognize/crnn_pseudo_ft.pt'
shutil.copy(WEIGHTS_OUT, dst)
print(f'Saved to Drive: {dst}')
print(f'Best govdoc CER: {best_cer:.4f}')
print()
print('Next steps:')
print('1. Download crnn_pseudo_ft.pt from Drive')
print('2. Place at: weights/recognize/crnn_pseudo_ft.pt')
print('3. In configs/pipeline.yaml set:')
print('     deva_backend: crnn_mozhi')
print('     crnn_mozhi:')
print('       weights: ./weights/recognize/crnn_pseudo_ft.pt')
print('4. Re-run: python eval/run_cer_comparison_with_wer.py')

In [ ]:
# ── Cell 10 (Optional): Quick visual sanity check ───────────────────────────
# Shows 5 govdoc crops with predicted vs GT text
import matplotlib.pyplot as plt

# Load best checkpoint
ckpt = torch.load(WEIGHTS_OUT, map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model.eval().to(DEVICE)

sample_rows = random.sample(all_govdoc, min(5, len(all_govdoc)))
fig, axes = plt.subplots(5, 1, figsize=(14, 10))

for ax, row in zip(axes, sample_rows):
    img = cv2.imread(row['image_path'], cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    new_w = min(int(w * IMG_H / h), MAX_W)
    img_r = cv2.resize(img, (new_w, IMG_H))
    if new_w < MAX_W:
        img_r = np.pad(img_r, ((0, 0), (0, MAX_W - new_w)), constant_values=255)
    t = torch.from_numpy(img_r).float().div(255).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        lp = model(t)
    pred = list(greedy_decode(lp, charset))[0][0]
    ax.imshow(img, cmap='gray', aspect='auto')
    ax.set_title(f'GT:   {row["text"]}\nPred: {pred}', fontsize=9, loc='left')
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to /content/sample_predictions.png')